# 10 - Ensemble: does the text add anything?

Notebooks 05 and 07 predict the price from the **structured features**, notebook 08 from the **listing text**.
Here we combine their predictions to answer one question:

> Does the text contain information that the structured features do not?

Each combination is a **weighted average** of the models' `log1p(price)` predictions.
The weights are learned on the validation set and the result is scored on the test set.
If adding a text model improves the test score, the text carries extra information.

This replaces training one large network on both inputs: it answers the same question
without a second, expensive PhoBERT fine-tuning.

Run notebooks 05, 07 and 08 first: they save their predictions to `reports/predictions/`.

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             r2_score, root_mean_squared_error)

## Data

PhoBERT only predicted a sample of rows (10k validation, 50k test), so we keep only those rows.
Every model is then compared on exactly the same listings.

In [ ]:
parts = [
    pd.read_parquet("../reports/predictions/05_tree_ensembles.parquet").drop(columns="split"),
    pd.read_parquet("../reports/predictions/07_deep_tabular.parquet").drop(columns="split"),
    pd.read_parquet("../reports/predictions/08_phobert.parquet").drop(columns="split"),
]
pred = pd.concat(parts, axis=1, join="inner")   # inner join = rows that every model predicted

data = pd.read_parquet("../data/processed/model_input.parquet", columns=["price", "split"])
pred = pred.join(data)

val = pred[pred["split"] == "val"]
test = pred[pred["split"] == "test"]

MODELS = ["LightGBM", "CatBoost", "MLP", "TF-IDF", "PhoBERT"]
pred["split"].value_counts()

## Combinations

| Combination | Models |
|---|---|
| tabular | LightGBM, CatBoost, MLP |
| tabular + TF-IDF | tabular + TF-IDF |
| tabular + PhoBERT | tabular + PhoBERT |
| all | all five |

The weights come from a linear regression of the true log price on the models' predictions (validation set).
`positive=True` keeps every weight at zero or above, so the result stays an average of the models.

In [ ]:
TABULAR = ["LightGBM", "CatBoost", "MLP"]
COMBINATIONS = {
    "Blend: tabular": TABULAR,
    "Blend: tabular + TF-IDF": TABULAR + ["TF-IDF"],
    "Blend: tabular + PhoBERT": TABULAR + ["PhoBERT"],
    "Blend: all": MODELS,
}


def evaluate(y_true, y_pred):
    return {
        "MAE (bn VND)": mean_absolute_error(y_true, y_pred) / 1e9,
        "RMSE (bn VND)": root_mean_squared_error(y_true, y_pred) / 1e9,
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": 100 * mean_absolute_percentage_error(y_true, y_pred),
    }


results, weights = {}, {}

# each model on its own
for name in MODELS:
    results[name] = evaluate(test["price"], np.expm1(test[name]))

# weighted averages
for name, columns in COMBINATIONS.items():
    blend = LinearRegression(positive=True).fit(val[columns], np.log1p(val["price"]))
    results[name] = evaluate(test["price"], np.expm1(blend.predict(test[columns])))
    weights[name] = pd.Series(blend.coef_, index=columns)

## Results

All rows are scored on the same test listings.

In [ ]:
table = pd.DataFrame(results).T

os.makedirs("../reports/metrics", exist_ok=True)
table.to_csv("../reports/metrics/09_ensemble.csv")

table.style.format("{:,.3f}")

### Weights

A model with a weight near 0 adds nothing that the other models in the blend do not already know.

In [ ]:
pd.DataFrame(weights).T.style.format("{:.2f}", na_rep="")

### How to read this

- Compare **"Blend: tabular"** with **"Blend: tabular + PhoBERT"** (and **+ TF-IDF**): the difference is what the text adds.
- Compare **"+ TF-IDF"** with **"+ PhoBERT"**: does the pretrained model find more in the text than word counts?
- The tabular models were trained on 1.7M listings and the text models on 100k, so the text models are at a disadvantage.
  A gain from adding them is therefore a conservative estimate.